In [10]:
# Import essential models and functions from sklearn
from sklearn.linear_model import LinearRegression, LogisticRegression 
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, StratifiedKFold
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt
sb.set()
np.random.seed(42)

In [11]:
d  = pd.read_csv('clean_data.csv')
d1 = pd.read_csv('owid-energy-data.csv')

candidate_features = [
    "population", "electricity_generation", "primary_energy_consumption",
    "fossil_share_energy", "energy_per_capita",
    "renewables_share_energy", "coal_share_energy", "oil_share_energy",
    "gas_share_energy", "low_carbon_share_energy", "renewables_share_elec"
]

years = sorted(d1["year"].unique())
year_top_corr = {}

for yr in years:
    year_slice = d1[d1["year"] == yr]
    avail_cols = [c for c in candidate_features if c in year_slice.columns]
    corr_df = year_slice[avail_cols + ["gdp"]].dropna()
    if corr_df.shape[0] == 0:
        year_top_corr[yr] = pd.Series(dtype=float)
        continue
    corr = corr_df.corr(numeric_only=True)["gdp"].drop("gdp")
    corr = corr.dropna().sort_values(key=lambda x: x.abs(), ascending=False)
    year_top_corr[yr] = corr

N_TOP = 8
rows = []
for yr, corr in year_top_corr.items():
    topN = corr.head(N_TOP)
    row = {"year": yr}
    for i in range(N_TOP):
        if i < len(topN):
            row[f"feature_{i+1}"] = topN.index[i]
            row[f"corr_{i+1}"] = round(topN.iloc[i], 4)
        else:
            row[f"feature_{i+1}"] = None
            row[f"corr_{i+1}"] = None
    rows.append(row)

year_top8_df = pd.DataFrame(rows).sort_values("year").reset_index(drop=True)
print(year_top8_df)

FileNotFoundError: [Errno 2] No such file or directory: 'clean_data.csv'

In [5]:
#G7 

g7_countries = ["United States", "Canada", "United Kingdom", "France", "Germany", "Italy", "Japan"]

d1 = d1.sort_values(["country", "year"])
d1["gdp_growth"] = d1.groupby("country")["gdp"].pct_change(fill_method=None)
d1["growth_median_year"] = d1.groupby("year")["gdp_growth"].transform("median")
d1["growth_label"] = np.where(d1["gdp_growth"] > d1["growth_median_year"], 1, 0)
d1.loc[d1["gdp_growth"].isna(), "growth_label"] = np.nan

g7_df = d1[d1["country"].isin(g7_countries)].copy()

g7_corr_by_country = {}
for country in g7_countries:
    sub = g7_df[g7_df["country"] == country]
    valid = sub[candidate_features + ["gdp"]].dropna()
    if valid.shape[0] < 5:
        continue
    corr = valid.corr(numeric_only=True)["gdp"].drop("gdp").dropna()
    g7_corr_by_country[country] = corr.sort_values(key=lambda x: x.abs(), ascending=False)

for country, corr in g7_corr_by_country.items():
    print(f"\n--- {country} ---")
    print(corr.head(5))


#different countries - different top1 predictors

NameError: name 'd1' is not defined

In [6]:
# VIF & labels of strong/weak electroenergies

g7_df["elec_median_year"] = g7_df.groupby("year")["electricity_generation"].transform("median")
g7_df["power_label"] = np.where(g7_df["electricity_generation"] > g7_df["elec_median_year"], 1, 0)

model_features = ["electricity_generation", "energy_per_capita"]
req_cols = model_features + ["power_label"]
g7_model_df = g7_df.dropna(subset=req_cols).copy()

X = g7_model_df[model_features].copy()
y_power  = g7_model_df["power_label"].astype(int)
y_gdp    = g7_model_df["gdp"].copy()

X_vif = add_constant(X)
vif_data = pd.DataFrame({
    "feature": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
print(vif_data[vif_data["feature"] != "const"].sort_values("VIF", ascending=False))

NameError: name 'g7_df' is not defined

In [7]:
g7_countries = ["United States", "Canada", "United Kingdom", "France", "Germany", "Italy", "Japan"]

d1 = pd.read_csv('owid-energy-data.csv')
d1 = d1.sort_values(["country", "year"])

g7_df = d1[d1["country"].isin(g7_countries)].copy()
g7_df["elec_median_year"] = g7_df.groupby("year")["electricity_generation"].transform("median")
g7_df["power_label"] = np.where(g7_df["electricity_generation"] > g7_df["elec_median_year"], 1, 0)

model_features = ["electricity_generation"]
req_cols = model_features + ["power_label", "gdp"]
g7_model_df = g7_df.dropna(subset=req_cols).copy()

X = g7_model_df[model_features].copy()
y_power = g7_model_df["power_label"].astype(int)
y_gdp   = g7_model_df["gdp"].copy()

X_train, X_test, ypw_train, ypw_test, ygdp_train, ygdp_test = train_test_split(
    X, y_power, y_gdp, test_size=0.25, random_state=42, stratify=y_power
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=model_features, index=X_train.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test), columns=model_features, index=X_test.index)

logreg_power = LogisticRegression(max_iter=1000, random_state=42)
logreg_power.fit(X_train_scaled, ypw_train)
pred_lg_power = logreg_power.predict(X_test_scaled)

dt_power = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_power.fit(X_train, ypw_train)
pred_dt_power = dt_power.predict(X_test)

linreg = LinearRegression()
linreg.fit(X_train, ygdp_train)
test_r2 = linreg.score(X_test, ygdp_test)
test_mse = mean_squared_error(ygdp_test, linreg.predict(X_test))

# ---- Confusion matrices: 1 строка, 2 столбца ----
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_lg_power = confusion_matrix(ypw_test, pred_lg_power)
ConfusionMatrixDisplay(cm_lg_power, display_labels=["Weak power","Strong power"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Logistic Regression — Power")

cm_dt_power = confusion_matrix(ypw_test, pred_dt_power)
ConfusionMatrixDisplay(cm_dt_power, display_labels=["Weak power","Strong power"]).plot(ax=axes[1], colorbar=False)
axes[1].set_title("Decision Tree — Power")

plt.tight_layout()
plt.savefig("g7_confusion_matrices_power_only.png", dpi=150)
plt.show()

print(f"Linear Regression GDP — Test R2: {test_r2:.4f}, Test MSE: {test_mse:.3e}")

FileNotFoundError: [Errno 2] No such file or directory: 'owid-energy-data.csv'

In [8]:
# Regression models and metrics
pred_gdp_test = linreg.predict(X_test)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(ygdp_test, pred_gdp_test, alpha=0.6, color="steelblue")
lims = [min(ygdp_test.min(), pred_gdp_test.min()), max(ygdp_test.max(), pred_gdp_test.max())]
axes[0].plot(lims, lims, 'r--', label="Perfect fit")
axes[0].set_xlabel("Actual GDP")
axes[0].set_ylabel("Predicted GDP")
axes[0].set_title(f"Linear Regression: Actual vs Predicted\nTest R2={test_r2:.3f}")
axes[0].legend()

residuals = ygdp_test - pred_gdp_test
axes[1].scatter(pred_gdp_test, residuals, alpha=0.6, color="darkorange")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_xlabel("Predicted GDP")
axes[1].set_ylabel("Residuals")
axes[1].set_title("Residual Plot")

plt.tight_layout()
plt.savefig("g7_linreg_diagnostics.png", dpi=150)
plt.close()

NameError: name 'linreg' is not defined

In [9]:
# summary
all_results = [
    {"Model":"Logistic Regression","Target":"Power","Accuracy":accuracy_score(ypw_test,pred_lg_power),
     "Precision":precision_score(ypw_test,pred_lg_power,zero_division=0),
     "Recall":recall_score(ypw_test,pred_lg_power,zero_division=0),
     "F1":f1_score(ypw_test,pred_lg_power,zero_division=0), "R2":None, "MSE":None},
    {"Model":"Decision Tree","Target":"Power","Accuracy":accuracy_score(ypw_test,pred_dt_power),
     "Precision":precision_score(ypw_test,pred_dt_power,zero_division=0),
     "Recall":recall_score(ypw_test,pred_dt_power,zero_division=0),
     "F1":f1_score(ypw_test,pred_dt_power,zero_division=0), "R2":None, "MSE":None},
    {"Model":"Linear Regression","Target":"GDP (regression)","Accuracy":None,"Precision":None,
     "Recall":None,"F1":None,"R2":test_r2,"MSE":test_mse},
]

full_results_df = pd.DataFrame(all_results).round(4)
full_results_df.to_csv("g7_all_models_metrics.csv", index=False)
print(full_results_df)

NameError: name 'ypw_test' is not defined